# Part 1: EDA

We began by examining the raw light curves directly to understand the basic structure of the data. The raw plots showed that flux magnitude varied dramatically across stars, with some series exhibiting large drifts, oscillations, or spikes that made direct comparison difficult. This confirmed that raw flux values were not on a common scale and that some stars contained substantial noise or artifact-like behavior.

To make stars comparable, we applied per-star z-score normalization and replotted the same examples. After normalization, differences in absolute scale were removed, making relative shape and dimming behavior much easier to compare. This step was important because our downstream engineered features were meant to capture *relative* transit-like behavior rather than raw brightness.

We also inspected the positive class more closely by plotting all exoplanet-labeled stars after normalization. This showed that the positive class was not uniform: some stars displayed repeated sharp dips, others showed smoother low-flux behavior, and some still contained spikes or irregularities. This suggested that confirmed exoplanet signals in the dataset are heterogeneous, which helps explain why class separation is only partial rather than clean.

Next, we created engineered feature distribution plots for several transit-motivated features. These showed that exoplanet-labeled stars tended to have:
- more negative low-tail behavior (`p01_z`, `min_z`),
- more time spent in unusually low flux (`frac_below_-2`),
- longer continuous low-flux runs (`longest_run_-1`),
- and lower local roughness (`mean_abs_diff`) than many negative examples.
These plots provided the first strong evidence that simple interpretable features could separate the classes.

Because some features were strongly skewed, we also used a skew-adjusted plot of `log1p(longest_run_-1)`. This made the run-length difference easier to see than the raw boxplot and confirmed that exoplanet stars generally had longer sustained low-flux episodes.

We then highlighted one strong single feature, `p01_z`, with a strip-over-box plot. This showed that the positive class generally had a deeper negative tail than the negative class, but with substantial overlap. This was useful for showing that individual features carry signal, while also making it clear that no single feature perfectly solves the task.

To quantify separation more directly, we computed feature effect size proxies comparing positives and negatives. This showed that the strongest engineered signals included:
- `frac_below_-2` and `longest_run_-1` (both larger for positives),
- `p01_z` (more negative for positives),
- and `mean_abs_diff` (smaller for positives).
This supported the interpretation that exoplanet stars often combine deeper, more sustained dimming with smoother local structure.

We also examined pairwise feature correlations to understand redundancy. The correlation heatmap showed clear groups of related features (for example, depth features, threshold-based dip-frequency features, and variability features), indicating that several engineered features were measuring similar aspects of the signal. This later motivated feature pruning and regularization in the inferential model.

A PCA projection of the engineered feature space showed that exoplanet and non-exoplanet stars were not cleanly separable. The positive class occupied a somewhat shifted region of the space, but there was still heavy overlap with the negative class. This reinforced that the task is nontrivial and justified the need for model-based classification rather than simple threshold rules alone.

We also checked robustness to preprocessing by comparing effect size behavior under z-score and robust (median/IQR) scaling. The key features kept the same directional trends under both methods, indicating that the main feature story was stable and not an artifact of one normalization choice.

Finally, we compared several important engineered features between train and test. The train/test histograms for `p01_z`, `frac_below_-2`, and `mean_abs_diff` were broadly similar, with no obvious large distribution shift. This supported the use of the provided train/test split and suggested that the engineered feature distributions were reasonably consistent across both sets.

## Main EDA takeaway

Overall, the EDA showed that the problem is difficult because the positive class is extremely small and overlaps heavily with the negative class, but also that exoplanet-labeled stars do display measurable differences in low-flux depth, low-flux duration, dip frequency, and local smoothness. These findings justified both the engineered-feature approach and the later use of regularized logistic regression and calibrated classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRAIN_PATH = "exoTrain.csv"
TEST_PATH  = "exoTest.csv"

def load_exo(path: str):
    df = pd.read_csv(path)
    y_raw = df["LABEL"].astype(int).to_numpy()
    y = (y_raw == 2).astype(int)  # 1=exoplanet, 0=non-exoplanet
    flux_cols = [c for c in df.columns if c.startswith("FLUX.")]
    X = df[flux_cols].to_numpy(dtype=np.float32)
    return X, y, y_raw, flux_cols

X_train, y_train, yraw_train, flux_cols = load_exo(TRAIN_PATH)
X_test,  y_test,  yraw_test,  _        = load_exo(TEST_PATH)

def per_series_zscore(X: np.ndarray, eps: float = 1e-8):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return (X - mu) / (sd + eps)

def per_series_robust(X: np.ndarray, eps: float = 1e-8):
    med = np.median(X, axis=1, keepdims=True)
    q75 = np.quantile(X, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(X, 0.25, axis=1, keepdims=True)
    iqr = q75 - q25
    return (X - med) / (iqr + eps)

In [ ]:
rng = np.random.default_rng(0)

def pick_indices(y, n_neg=8, n_pos=8):
    neg_idx = rng.choice(np.where(y==0)[0], size=n_neg, replace=False)
    pos_pool = np.where(y==1)[0]
    pos_idx = rng.choice(pos_pool, size=min(n_pos, len(pos_pool)), replace=False)
    return neg_idx, pos_idx

neg_idx, pos_idx = pick_indices(y_train, n_neg=8, n_pos=8)

Z_train = per_series_zscore(X_train)

def plot_lightcurves_panel(X_raw, X_norm, neg_idx, pos_idx, title_prefix="Train"):
    t = np.arange(X_raw.shape[1])

    fig, axes = plt.subplots(2, 2, figsize=(12,6), sharex=True)
    axes = axes.ravel()

    # raw negatives
    for i in neg_idx:
        axes[0].plot(t, X_raw[i], linewidth=1)
    axes[0].set_title(f"{title_prefix}: Non-exoplanet (raw)")

    # raw positives
    for i in pos_idx:
        axes[1].plot(t, X_raw[i], linewidth=1)
    axes[1].set_title(f"{title_prefix}: Exoplanet (raw)")

    # normalized negatives
    for i in neg_idx:
        axes[2].plot(t, X_norm[i], linewidth=1)
    axes[2].set_title(f"{title_prefix}: Non-exoplanet (z-score)")

    # normalized positives
    for i in pos_idx:
        axes[3].plot(t, X_norm[i], linewidth=1)
    axes[3].set_title(f"{title_prefix}: Exoplanet (z-score)")

    for ax in axes:
        ax.set_xlabel("time index")
        ax.set_ylabel("flux")
    plt.tight_layout()
    plt.savefig(f"./plots/lightcurves_panel.png")
    plt.close()

plot_lightcurves_panel(X_train, Z_train, neg_idx, pos_idx, "Train")

In [ ]:
def longest_run_below(z_row, thr=-1.0):
    below = (z_row < thr)
    # compute run lengths
    max_run = 0
    run = 0
    for b in below:
        if b:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def num_runs_below(z_row, thr=-1.0):
    below = (z_row < thr)
    # count transitions False->True
    return int(np.sum((~below[:-1]) & (below[1:])) + (1 if below[0] else 0))

def extract_features(X):
    Z = per_series_zscore(X)
    feats = []
    for z in Z:
        diff = np.diff(z)
        p01 = np.quantile(z, 0.01)
        p05 = np.quantile(z, 0.05)
        feats.append({
            "min_z": float(np.min(z)),
            "p01_z": float(p01),
            "p05_z": float(p05),
            "frac_below_-1": float(np.mean(z < -1)),
            "frac_below_-2": float(np.mean(z < -2)),
            "frac_below_-3": float(np.mean(z < -3)),
            "longest_run_-1": float(longest_run_below(z, -1)),
            "num_runs_-1": float(num_runs_below(z, -1)),
            "mad_z": float(np.median(np.abs(z - np.median(z)))),
            "iqr_z": float(np.quantile(z, 0.75) - np.quantile(z, 0.25)),
            "mean_abs_diff": float(np.mean(np.abs(diff))),
            "std_diff": float(np.std(diff)),
        })
    return pd.DataFrame(feats)

feat_tr = extract_features(X_train)

In [ ]:
def boxplots_by_class(feat_df, y, cols, title):
    fig, axes = plt.subplots(2, 3, figsize=(12,6))
    axes = axes.ravel()
    for ax, col in zip(axes, cols):
        ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
        ax.set_title(col)
        ax.set_xlabel("class (0=non, 1=exo)")
    fig.suptitle(title)
    plt.savefig(f"./plots/engineered_features_box.png")
    plt.close()

key_cols = ["p01_z","frac_below_-2","longest_run_-1","mean_abs_diff","mad_z","min_z"]
boxplots_by_class(feat_tr, y_train, key_cols, "Engineered feature distributions (train)")

def strip_over_box(feat_df, y, col, title):
    x0 = np.zeros(np.sum(y==0))
    x1 = np.ones(np.sum(y==1))
    fig, ax = plt.subplots(figsize=(6,4))
    ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], positions=[0,1], widths=0.5)
    ax.scatter(x0 + 0.05*rng.standard_normal(len(x0)), feat_df.loc[y==0, col], s=10, alpha=0.4)
    ax.scatter(x1 + 0.05*rng.standard_normal(len(x1)), feat_df.loc[y==1, col], s=25, alpha=0.9)
    ax.set_xticks([0,1]); ax.set_xticklabels(["0","1"])
    ax.set_title(title); ax.set_xlabel("class"); ax.set_ylabel(col)
    plt.savefig(f"./plots/p01_z_strip_over_box.png")
    plt.close()

strip_over_box(feat_tr, y_train, "p01_z", "p01_z by class (train)")

C:\Users\haesu\AppData\Local\Temp\ipykernel_25392\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_25392\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_25392\1657152811.py:5: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([feat_df.loc[y==0, col], feat_df.loc[y==1, col]], labels=["0", "1"])
C:\Users\haesu\AppData\Local\Temp\ipykernel_25

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

corr = feat_tr.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8,6))
im = ax.imshow(corr.values, aspect="auto")
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation among engineered features (train)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.savefig("./plots/feature_correlation.png")
plt.close()

Xf = feat_tr.to_numpy()
Xf_std = StandardScaler().fit_transform(Xf)
pca = PCA(n_components=2, random_state=0)
X2 = pca.fit_transform(Xf_std)

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(X2[y_train==0,0], X2[y_train==0,1], s=10, alpha=0.4, label="0")
ax.scatter(X2[y_train==1,0], X2[y_train==1,1], s=30, alpha=0.9, label="1")
ax.set_title("PCA of engineered features (train)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend()
plt.savefig("./plots/feature_pca.png")
plt.close()

In [ ]:
# ---------- scaling helpers ----------
def per_series_zscore(X: np.ndarray, eps: float = 1e-8):
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    return (X - mu) / (sd + eps)

def per_series_robust(X: np.ndarray, eps: float = 1e-8):
    med = np.median(X, axis=1, keepdims=True)
    q75 = np.quantile(X, 0.75, axis=1, keepdims=True)
    q25 = np.quantile(X, 0.25, axis=1, keepdims=True)
    iqr = q75 - q25
    return (X - med) / (iqr + eps)

# ---------- run-length helpers ----------
def longest_run_below(z_row, thr=-1.0):
    below = (z_row < thr)
    max_run = 0
    run = 0
    for b in below:
        if b:
            run += 1
            max_run = max(max_run, run)
        else:
            run = 0
    return max_run

def num_runs_below(z_row, thr=-1.0):
    below = (z_row < thr)
    return int(np.sum((~below[:-1]) & (below[1:])) + (1 if below[0] else 0))

# ---------- feature extractor ----------
def extract_features_scaled(X: np.ndarray, scale: str = "zscore") -> pd.DataFrame:
    if scale == "zscore":
        Z = per_series_zscore(X)
    elif scale == "robust":
        Z = per_series_robust(X)
    else:
        raise ValueError("scale must be 'zscore' or 'robust'")

    feats = []
    for z in Z:
        diff = np.diff(z)
        feats.append({
            "min_z": float(np.min(z)),
            "p01_z": float(np.quantile(z, 0.01)),
            "p05_z": float(np.quantile(z, 0.05)),
            "frac_below_-1": float(np.mean(z < -1)),
            "frac_below_-2": float(np.mean(z < -2)),
            "frac_below_-3": float(np.mean(z < -3)),
            "longest_run_-1": float(longest_run_below(z, -1)),
            "num_runs_-1": float(num_runs_below(z, -1)),
            "mad_z": float(np.median(np.abs(z - np.median(z)))),
            "iqr_z": float(np.quantile(z, 0.75) - np.quantile(z, 0.25)),
            "mean_abs_diff": float(np.mean(np.abs(diff))),
            "std_diff": float(np.std(diff)),
            "acf1": float(np.corrcoef(z[:-1], z[1:])[0, 1]) if np.std(z[:-1]) > 0 and np.std(z[1:]) > 0 else 0.0,
            "acf2": float(np.corrcoef(z[:-2], z[2:])[0, 1]) if np.std(z[:-2]) > 0 and np.std(z[2:]) > 0 else 0.0,
        })
    return pd.DataFrame(feats)

feat_tr_z = extract_features_scaled(X_train, scale="zscore")
feat_te_z = extract_features_scaled(X_test,  scale="zscore")

feat_tr_r = extract_features_scaled(X_train, scale="robust")
feat_te_r = extract_features_scaled(X_test,  scale="robust")

In [ ]:
def summarize_feature_differences(feat_df: pd.DataFrame, y: np.ndarray, cols):
    rows = []
    for col in cols:
        neg = feat_df.loc[y == 0, col].to_numpy()
        pos = feat_df.loc[y == 1, col].to_numpy()

        med_neg = np.median(neg)
        med_pos = np.median(pos)
        mean_neg = np.mean(neg)
        mean_pos = np.mean(pos)

        # simple pooled-std effect size proxy
        pooled_std = np.sqrt((np.var(neg, ddof=1) + np.var(pos, ddof=1)) / 2)
        effect = (mean_pos - mean_neg) / (pooled_std + 1e-8)

        rows.append({
            "feature": col,
            "median_neg": med_neg,
            "median_pos": med_pos,
            "median_diff": med_pos - med_neg,
            "mean_neg": mean_neg,
            "mean_pos": mean_pos,
            "effect_size_proxy": effect
        })

    out = pd.DataFrame(rows)
    out["abs_effect"] = out["effect_size_proxy"].abs()
    out = out.sort_values("abs_effect", ascending=False).drop(columns="abs_effect")
    return out

key_cols = ["p01_z", "frac_below_-2", "longest_run_-1", "mean_abs_diff", "mad_z", "min_z"]
feature_summary = summarize_feature_differences(feat_tr_z, y_train, key_cols)
print(feature_summary)

def plot_feature_effect_sizes(summary_df: pd.DataFrame, title="Feature differences by class"):
    df = summary_df.copy().sort_values("effect_size_proxy")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.barh(df["feature"], df["effect_size_proxy"])
    ax.axvline(0, linestyle="--", linewidth=1)
    ax.set_xlabel("Effect size proxy (mean_pos - mean_neg) / pooled std")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f"./plots/feature_effect_sizes.png")
    plt.close()

plot_feature_effect_sizes(feature_summary, "Feature effect size proxies (train)")

          feature  median_neg  median_pos  median_diff   mean_neg   mean_pos  \
3   mean_abs_diff    0.423039    0.243727    -0.179313   0.430096   0.263730   
0           p01_z   -2.082875   -2.469438    -0.386562  -2.097199  -2.852652   
1   frac_below_-2    0.012199    0.022208     0.010009   0.013778   0.021312   
2  longest_run_-1   16.000000   24.000000     8.000000  33.235446  93.216216   
4           mad_z    0.508674    0.582607     0.073933   0.494015   0.540794   
5           min_z   -3.794927   -4.310022    -0.515095  -5.486261  -5.264079   

   effect_size_proxy  
3          -0.942935  
0          -0.738829  
1           0.618549  
2           0.553805  
4           0.262549  
5           0.052974  


In [ ]:
def plot_skewed_feature_log_box(feat_df: pd.DataFrame, y: np.ndarray, col="longest_run_-1"):
    vals0 = np.log1p(feat_df.loc[y == 0, col].to_numpy())
    vals1 = np.log1p(feat_df.loc[y == 1, col].to_numpy())

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot([vals0, vals1], labels=["Non-exoplanet (0)", "Exoplanet (1)"])
    ax.set_ylabel(f"log1p({col})")
    ax.set_title(f"Skew-adjusted distribution of {col}")
    plt.tight_layout()
    plt.savefig(f"./plots/longest_run_-1_box.png")
    plt.close()

plot_skewed_feature_log_box(feat_tr_z, y_train, "longest_run_-1")

C:\Users\haesu\AppData\Local\Temp\ipykernel_25392\2468572731.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([vals0, vals1], labels=["Non-exoplanet (0)", "Exoplanet (1)"])


In [ ]:
compare_cols = ["p01_z", "frac_below_-2", "mean_abs_diff"]

def compare_scaling_effects(feat_tr_z, feat_tr_r, y, cols):
    zsum = summarize_feature_differences(feat_tr_z, y, cols).set_index("feature")
    rsum = summarize_feature_differences(feat_tr_r, y, cols).set_index("feature")

    comp = pd.DataFrame({
        "zscore": zsum.loc[cols, "effect_size_proxy"],
        "robust": rsum.loc[cols, "effect_size_proxy"]
    })

    x = np.arange(len(cols))
    w = 0.35

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(x - w/2, comp["zscore"], w, label="z-score")
    ax.bar(x + w/2, comp["robust"], w, label="robust")
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(cols, rotation=30, ha="right")
    ax.set_ylabel("Effect size proxy")
    ax.set_title("Feature effect sizes under z-score vs robust scaling")
    ax.legend()
    plt.tight_layout()
    plt.savefig("./plots/scaling_effect_size_comparison.png")
    plt.close()

compare_scaling_effects(feat_tr_z, feat_tr_r, y_train, compare_cols)

In [ ]:
def plot_all_positive_lightcurves(X: np.ndarray, y: np.ndarray, sort_by=None, feat_df=None, ncols=4):
    Z = per_series_zscore(X)
    pos_idx = np.where(y == 1)[0]

    if sort_by is not None and feat_df is not None:
        order = np.argsort(feat_df.loc[pos_idx, sort_by].to_numpy())
        pos_idx = pos_idx[order]

    n = len(pos_idx)
    nrows = int(np.ceil(n / ncols))
    t = np.arange(X.shape[1])

    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.5 * nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    for ax, idx in zip(axes, pos_idx):
        ax.plot(t, Z[idx], linewidth=0.8)
        title = f"idx={idx}"
        if sort_by is not None and feat_df is not None:
            title += f"\n{sort_by}={feat_df.loc[idx, sort_by]:.2f}"
        ax.set_title(title, fontsize=8)
        ax.set_xlabel("time")
        ax.set_ylabel("z")

    # hide unused axes
    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle("All positive-class light curves (z-scored)", y=1.01)
    plt.tight_layout()
    plt.savefig("./plots/all_positive_lightcurves.png")
    plt.close()

plot_all_positive_lightcurves(X_train, y_train, sort_by="p01_z", feat_df=feat_tr_z, ncols=4)

In [ ]:
def overlay_train_test_hist(feat_train: pd.DataFrame, feat_test: pd.DataFrame, col: str, bins=40):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(feat_train[col], bins=bins, alpha=0.6, label="Train")
    ax.hist(feat_test[col],  bins=bins, alpha=0.6, label="Test")
    ax.set_title(f"Train vs test distribution: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()
    plt.savefig(f"./plots/train_test_{col}_hist.png")
    plt.close()

for col in ["p01_z", "frac_below_-2", "mean_abs_diff"]:
    overlay_train_test_hist(feat_tr_z, feat_te_z, col)

# Part 2: Feature selection and preprocessing

We finalized per-star z-score normalization as the main preprocessing step. This was chosen because raw flux values varied dramatically across stars, making absolute magnitude unreliable for comparison. Z-scoring preserved relative light-curve shape and dimming behavior, which is what our engineered features were meant to capture.

We also compared z-score against robust scaling (median/IQR) as a sanity check. The key feature trends kept the same directional behavior under both methods, so we kept z-score for simplicity and consistency.

From the broader engineered feature pool, we first examined pairwise correlations to identify redundancy. This showed several near-duplicate groups, such as `mad_z` with `iqr_z`, `acf1` with `acf2`, and strong overlap among the threshold-based dip-frequency features. Based on this, we pruned to a smaller candidate set that kept one representative from each correlated group.

We then ranked features by a simple effect size proxy comparing exoplanet vs non-exoplanet classes. This showed that both local structure/smoothness features (`acf1`, `mean_abs_diff`, `std_diff`) and transit-motivated dip features (`p01_z`, `frac_below_-2`, `longest_run_-1`) carried signal.

To make the selection less subjective, we ran L1-regularized logistic regression with 5-fold stratified cross-validation as a stability check. Features consistently selected across folds were treated as the strongest candidates.

The features selected in all 5 folds were: `acf1`, `mad_z`, `longest_run_-1`, `frac_below_-2`, `p01_z`, and `mean_abs_diff`. By contrast, `std_diff` and `num_runs_-1` were selected inconsistently, so they were excluded from the final core set.

This left us with a compact final feature set that captures:
- short-lag structure / smoothness: `acf1`, `mean_abs_diff`
- overall robust variability: `mad_z`
- depth, frequency, and duration of low-flux behavior: `p01_z`, `frac_below_-2`, `longest_run_-1`

Overall, the feature selection process supported the idea that exoplanet-labeled light curves differ not only by having deeper or longer low-flux excursions, but also by exhibiting smoother, more structured local behavior than many negative examples.

In [ ]:
candidate_cols = [
    "min_z",
    "p01_z",
    "p05_z",
    "frac_below_-1",
    "frac_below_-2",
    "frac_below_-3",
    "longest_run_-1",
    "num_runs_-1",
    "mad_z",
    "iqr_z",
    "mean_abs_diff",
    "std_diff",
    "acf1",
    "acf2",
]

candidate_df = feat_tr_z[candidate_cols].copy()
candidate_df.head()

,min_z,p01_z,p05_z,frac_below_-1,frac_below_-2,frac_below_-3,longest_run_-1,num_runs_-1,mad_z,iqr_z,mean_abs_diff,std_diff,acf1,acf2
0,-6.620421,-2.143195,-1.532662,0.154833,0.015014,0.001877,24.0,147.0,0.690029,1.381925,0.351541,0.551851,0.847769,0.721402
1,-6.063865,-4.754909,-1.277299,0.060369,0.037222,0.029403,11.0,56.0,0.390156,0.776812,0.332721,0.634420,0.798803,0.590351
2,-3.133562,-2.542234,-1.816857,0.115421,0.035346,0.001877,52.0,36.0,0.504307,1.021813,0.147196,0.298306,0.955496,0.912262
3,-2.555287,-2.162253,-1.670249,0.177041,0.026275,0.000000,198.0,54.0,0.706338,1.430985,0.128399,0.203164,0.979337,0.967220
4,-4.408515,-2.462599,-1.842500,0.162652,0.028151,0.002502,169.0,44.0,0.728054,1.433434,0.126879,0.303894,0.953776,0.909577


In [ ]:
def high_correlation_pairs(df: pd.DataFrame, threshold: float = 0.80) -> pd.DataFrame:
    corr = df.corr(numeric_only=True)
    rows = []
    cols = corr.columns.tolist()

    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            val = corr.iloc[i, j]
            if abs(val) >= threshold:
                rows.append({
                    "feature_1": cols[i],
                    "feature_2": cols[j],
                    "correlation": val
                })

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.reindex(out["correlation"].abs().sort_values(ascending=False).index)
    return out

high_corr = high_correlation_pairs(candidate_df, threshold=0.80)
print(high_corr.to_string(index=False))

    feature_1     feature_2  correlation
        mad_z         iqr_z     0.986815
     std_diff          acf1    -0.970432
     std_diff          acf2    -0.954342
frac_below_-1         mad_z     0.938715
         acf1          acf2     0.938013
frac_below_-1         iqr_z     0.934442
        p05_z frac_below_-1    -0.867976
  num_runs_-1 mean_abs_diff     0.860835
        p05_z         mad_z    -0.812535
        p05_z         iqr_z    -0.804235


In [ ]:
def summarize_feature_differences(feat_df: pd.DataFrame, y: np.ndarray, cols):
    rows = []
    for col in cols:
        neg = feat_df.loc[y == 0, col].to_numpy()
        pos = feat_df.loc[y == 1, col].to_numpy()

        mean_neg = np.mean(neg)
        mean_pos = np.mean(pos)
        med_neg = np.median(neg)
        med_pos = np.median(pos)

        pooled_std = np.sqrt((np.var(neg, ddof=1) + np.var(pos, ddof=1)) / 2)
        effect = (mean_pos - mean_neg) / (pooled_std + 1e-8)

        rows.append({
            "feature": col,
            "mean_neg": mean_neg,
            "mean_pos": mean_pos,
            "median_neg": med_neg,
            "median_pos": med_pos,
            "effect_size_proxy": effect,
            "abs_effect": abs(effect),
        })

    out = pd.DataFrame(rows).sort_values("abs_effect", ascending=False)
    return out.drop(columns="abs_effect")

feature_signal = summarize_feature_differences(candidate_df, y_train, candidate_cols)
print(feature_signal.to_string(index=False))

       feature   mean_neg  mean_pos  median_neg  median_pos  effect_size_proxy
          acf1   0.610794  0.837840    0.658233    0.847769           1.088409
      std_diff   0.819916  0.507450    0.825770    0.551851          -1.059589
          acf2   0.503501  0.733196    0.501434    0.745301           0.976957
 mean_abs_diff   0.430096  0.263730    0.423039    0.243727          -0.942935
         p01_z  -2.097199 -2.852652   -2.082875   -2.469438          -0.738829
   num_runs_-1 123.426931 76.513514  115.000000   72.000000          -0.692149
 frac_below_-2   0.013778  0.021312    0.012199    0.022208           0.618549
longest_run_-1  33.235446 93.216216   16.000000   24.000000           0.553805
 frac_below_-3   0.003471  0.006501    0.001251    0.003128           0.447499
 frac_below_-1   0.099378  0.122133    0.103222    0.137942           0.413875
         p05_z  -1.280100 -1.397753   -1.345194   -1.532662          -0.281936
         mad_z   0.494015  0.540794    0.508674    0

In [ ]:
# First-pass pruned set based on correlation groups + interpretability
pruned_cols = [
    "p01_z",          # representative of low-tail depth
    "frac_below_-2",  # representative of deep-dip frequency
    "longest_run_-1", # representative of dip duration
    "mean_abs_diff",  # representative of roughness
    "mad_z",          # representative of robust dispersion
    "acf1",           # optional structure feature
    "std_diff",       # optional secondary roughness feature
    "num_runs_-1",    # optional secondary dip-count feature
]

pruned_df = feat_tr_z[pruned_cols].copy()
print("Pruned feature set:")
print(pruned_df.columns.tolist())

Pruned feature set:
['p01_z', 'frac_below_-2', 'longest_run_-1', 'mean_abs_diff', 'mad_z', 'acf1', 'std_diff', 'num_runs_-1']


In [ ]:
scaler = StandardScaler()
X_pruned_scaled = scaler.fit_transform(pruned_df)

print("Scaled feature matrix shape:", X_pruned_scaled.shape)

Scaled feature matrix shape: (5087, 8)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

def l1_feature_selection_stability(X, y, feature_names, C=0.1, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    coef_rows = []
    selected_counts = pd.Series(0, index=feature_names, dtype=int)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, y_tr = X[train_idx], y[train_idx]

        clf = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=C,
            class_weight="balanced",
            max_iter=5000,
            random_state=random_state
        )
        clf.fit(X_tr, y_tr)

        coefs = pd.Series(clf.coef_[0], index=feature_names, name=f"fold_{fold}")
        coef_rows.append(coefs)

        selected_counts += (coefs != 0).astype(int)

    coef_df = pd.concat(coef_rows, axis=1)
    summary = pd.DataFrame({
        "feature": feature_names,
        "selected_in_folds": selected_counts.values,
        "selection_rate": selected_counts.values / n_splits,
        "mean_coef": coef_df.mean(axis=1).values,
        "std_coef": coef_df.std(axis=1).values,
    }).sort_values(["selection_rate", "selected_in_folds", "mean_coef"], ascending=[False, False, False])

    return coef_df, summary

coef_df, coef_summary = l1_feature_selection_stability(
    X_pruned_scaled,
    y_train,
    feature_names=pruned_cols,
    C=0.1,
    n_splits=5
)

print(coef_summary.to_string(index=False))

       feature  selected_in_folds  selection_rate  mean_coef  std_coef
          acf1                  5             1.0   0.972302  1.042215
         mad_z                  5             1.0   0.231584  0.177184
longest_run_-1                  5             1.0   0.102139  0.082379
 frac_below_-2                  5             1.0   0.045360  0.126105
         p01_z                  5             1.0  -0.722141  0.107756
 mean_abs_diff                  5             1.0  -0.799903  0.455805
      std_diff                  3             0.6   0.469355  0.445929
   num_runs_-1                  2             0.4   0.130698  0.197219


c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\haesu\Desktop\Stuff\exoplanet-hunting\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', 

In [ ]:
final_cols = coef_summary.loc[coef_summary["selected_in_folds"] >= 3, "feature"].tolist()

# If too many survive, keep top 5-6 by selection_rate then abs(mean_coef)
if len(final_cols) > 6:
    tmp = coef_summary.copy()
    tmp["abs_mean_coef"] = tmp["mean_coef"].abs()
    final_cols = (
        tmp.sort_values(["selection_rate", "abs_mean_coef"], ascending=[False, False])
           .head(6)["feature"]
           .tolist()
    )

print("Final selected features:")
print(final_cols)

Final selected features:
['acf1', 'mean_abs_diff', 'p01_z', 'mad_z', 'longest_run_-1', 'frac_below_-2']


# Part 3: Final modeling protocol setup

We finalized the modeling dataset using the six selected engineered features: `acf1`, `mean_abs_diff`, `p01_z`, `mad_z`, `longest_run_-1`, and `frac_below_-2`. This fixed the feature space before inferential or predictive modeling so that later comparisons would all use the same inputs.

The resulting feature matrices had shapes (5087, 6) for training and (570, 6) for test, confirming that the downstream models would operate on a compact, low-dimensional representation rather than the full raw light-curve sequence.

We kept both unscaled and standardized versions of the final feature matrix. Standardization was applied using statistics fit on the training set only, and the scaled results had near-zero means and unit variance across all six features, as expected. This standardized representation is especially important for logistic regression so that coefficient magnitudes are comparable and regularization behaves properly.

To make validation stable and fair under extreme class imbalance, we fixed a 5-fold stratified cross-validation protocol on the training set. The folds were balanced as well as possible given the small number of positives, with validation folds containing 7–8 positive examples each. This is a critical constraint of the dataset: each fold still contains very few positives, so performance estimates are expected to have noticeable variance.

The provided test set was left fully held out during this stage. This means all feature selection, coefficient analysis, model comparison, calibration, and threshold tuning can be done using the training set and its fixed CV structure, with the test set reserved only for final evaluation.

We also recorded baseline reference values before fitting any models. The training positive rate was 0.727%, and the test positive rate was 0.877%. An always-negative classifier would therefore achieve 99.27% accuracy on train and 99.12% accuracy on test, which reinforces that raw accuracy is not meaningful for this problem. Similarly, the positive base rate (~0.7%) serves as a “random precision” baseline: a useful model should substantially exceed this precision level.

Finally, we defined the reusable evaluation framework in advance. We set up metric helpers for:
- probability-based evaluation (`PR-AUC`, `ROC-AUC`)
- threshold-based evaluation (precision, recall, F1, confusion matrix)
- and threshold search under a minimum precision constraint (e.g. maximizing recall while maintaining precision ≥ 0.10).

Overall, this step froze the modeling protocol before any inferential or predictive results were generated. That reduces the risk of changing folds, metrics, or preprocessing mid-analysis and makes the later comparisons across models directly comparable.

## Main takeaway

* At this point, the experimental setup is fixed: the feature set, preprocessing pipeline, validation folds, and evaluation logic are all locked in. This means the next stages—interpreting coefficients in the inferential model and comparing predictive models—can focus entirely on model behavior rather than setup decisions.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, average_precision_score, roc_auc_score

# Final selected feature set from Part 2
final_cols = [
    "acf1",
    "mean_abs_diff",
    "p01_z",
    "mad_z",
    "longest_run_-1",
    "frac_below_-2",
]

# Build final train/test feature matrices
X_train_final = feat_tr_z[final_cols].copy()
X_test_final  = feat_te_z[final_cols].copy()

print("Final feature columns:")
print(final_cols)
print()

print("Train feature matrix shape:", X_train_final.shape)
print("Test feature matrix shape :", X_test_final.shape)
print()

# --------------------------------------------------
# Unscaled and scaled versions
# --------------------------------------------------

# Keep raw tabular feature values (useful for tree-based models)
X_train_unscaled = X_train_final.to_numpy(dtype=float)
X_test_unscaled  = X_test_final.to_numpy(dtype=float)

# Standardized version (fit on train only)
scaler_final = StandardScaler()
X_train_scaled = scaler_final.fit_transform(X_train_unscaled)
X_test_scaled  = scaler_final.transform(X_test_unscaled)

print("Scaled train matrix shape:", X_train_scaled.shape)
print("Scaled test matrix shape :", X_test_scaled.shape)
print()

# Optional: inspect feature means/stds after scaling
scaled_df = pd.DataFrame(X_train_scaled, columns=final_cols)
print("Approx. scaled train feature means:")
print(scaled_df.mean().round(4).to_string())
print()
print("Approx. scaled train feature stds:")
print(scaled_df.std(ddof=0).round(4).to_string())
print()

# --------------------------------------------------
# Fixed cross-validation protocol
# --------------------------------------------------

n_splits = 5
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

cv_splits = list(cv.split(X_train_scaled, y_train))

print(f"Number of CV folds: {len(cv_splits)}")
for i, (tr_idx, val_idx) in enumerate(cv_splits, start=1):
    y_tr_fold = y_train[tr_idx]
    y_val_fold = y_train[val_idx]
    print(
        f"Fold {i}: "
        f"train size={len(tr_idx)}, val size={len(val_idx)}, "
        f"train positives={y_tr_fold.sum()}, val positives={y_val_fold.sum()}"
    )

train_pos_rate = float(np.mean(y_train))
test_pos_rate = float(np.mean(y_test))

always_negative_train_acc = float(np.mean(y_train == 0))
always_negative_test_acc = float(np.mean(y_test == 0))

print("Baseline reference values")
print(f"Train positive rate: {train_pos_rate:.6f} ({train_pos_rate*100:.3f}%)")
print(f"Test positive rate : {test_pos_rate:.6f} ({test_pos_rate*100:.3f}%)")
print(f"Always-negative train accuracy: {always_negative_train_acc:.6f}")
print(f"Always-negative test accuracy : {always_negative_test_acc:.6f}")
print(f"Random precision baseline (train) ~= positive rate = {train_pos_rate:.6f}")
print()

# --------------------------------------------------
# Metric helper functions for later sections
# --------------------------------------------------

def compute_threshold_metrics(y_true, y_prob, threshold=0.5):
    """
    Compute threshold-based metrics from predicted probabilities.
    """
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }
    return out

def compute_prob_metrics(y_true, y_prob):
    """
    Compute probability-based metrics that do not require a threshold.
    """
    out = {
        "pr_auc": average_precision_score(y_true, y_prob),
    }

    # ROC-AUC is only defined if both classes are present
    if len(np.unique(y_true)) == 2:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    else:
        out["roc_auc"] = np.nan

    return out

def find_threshold_for_min_precision(y_true, y_prob, min_precision=0.10, n_steps=1000):
    """
    Find the threshold that maximizes recall while keeping precision >= min_precision.
    Simple grid search over [0, 1].
    """
    thresholds = np.linspace(0.0, 1.0, n_steps)
    best = None

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if prec >= min_precision:
            candidate = {
                "threshold": t,
                "precision": prec,
                "recall": rec,
                "f1": f1,
            }

            if best is None or candidate["recall"] > best["recall"]:
                best = candidate

    return best

print("Metric helpers ready:")
print("- compute_prob_metrics(y_true, y_prob)")
print("- compute_threshold_metrics(y_true, y_prob, threshold=...)")
print("- find_threshold_for_min_precision(y_true, y_prob, min_precision=0.10)")

Final feature columns:
['acf1', 'mean_abs_diff', 'p01_z', 'mad_z', 'longest_run_-1', 'frac_below_-2']

Train feature matrix shape: (5087, 6)
Test feature matrix shape : (570, 6)

Scaled train matrix shape: (5087, 6)
Scaled test matrix shape : (570, 6)

Approx. scaled train feature means:
acf1              0.0
mean_abs_diff     0.0
p01_z             0.0
mad_z            -0.0
longest_run_-1    0.0
frac_below_-2     0.0

Approx. scaled train feature stds:
acf1              1.0
mean_abs_diff     1.0
p01_z             1.0
mad_z             1.0
longest_run_-1    1.0
frac_below_-2     1.0

Number of CV folds: 5
Fold 1: train size=4069, val size=1018, train positives=29, val positives=8
Fold 2: train size=4069, val size=1018, train positives=29, val positives=8
Fold 3: train size=4070, val size=1017, train positives=30, val positives=7
Fold 4: train size=4070, val size=1017, train positives=30, val positives=7
Fold 5: train size=4070, val size=1017, train positives=30, val positives=7

Baselin